# Win Probability Model — Sanity Check

Run each cell to see the model's predicted win probability for real-world football scenarios.
Each scenario includes an **expected range** based on football intuition so you can judge whether the model makes sense.

**State convention:** all numbers are from the **possession team's** perspective.
- `score_differential` = (possession team score) − (defense team score)
- `yardline_100` = yards to the end zone (1 = goal line, 99 = own end zone)
- `seconds_remaining` counts down from 3600 (full game) to 0

In [3]:
import sys
sys.path.insert(0, '..')   # make src imports work from notebooks/

from src.models.win_probability import load_wp_model

model = load_wp_model()
print('Model loaded.')

Model loaded.


In [4]:
def check(label, state, expected_lo, expected_hi, note=''):
    """Predict WP and print a pass/fail line against an expected range."""
    wp = model.predict_proba(state)
    pct = wp * 100
    ok = expected_lo <= wp <= expected_hi
    status = '✓ PASS' if ok else '✗ FAIL'
    exp_str = f'[{expected_lo*100:.0f}%–{expected_hi*100:.0f}%]'
    print(f'{status}  {pct:5.1f}%  expected {exp_str:12s}  {label}')
    if note:
        print(f'        note: {note}')
    return wp

---
## 1. Blowout situations
These are the sanity check extremes — the model should be very confident.

In [5]:
print('=== Blowout situations ===\n')

# Up 28 with 2 minutes left — game is essentially over
check(
    'Up 28, 2:00 left Q4, own 20',
    dict(score_differential=28, quarter=4, seconds_remaining=120,
         yardline_100=80, down=1, ydstogo=10,
         offense_timeouts=2, defense_timeouts=3,
         is_overtime=0, overtime_possession_number=0),
    expected_lo=0.97, expected_hi=1.00,
    note='Should be >97% — three score lead, clock nearly dead'
)

# Down 28 with 2 minutes left — almost certainly losing
check(
    'Down 28, 2:00 left Q4, own 20',
    dict(score_differential=-28, quarter=4, seconds_remaining=120,
         yardline_100=80, down=1, ydstogo=10,
         offense_timeouts=3, defense_timeouts=2,
         is_overtime=0, overtime_possession_number=0),
    expected_lo=0.00, expected_hi=0.03,
    note='Should be <3% — need three scores in two minutes, essentially impossible'
)

# Up 14 with 5 minutes left — strong but not guaranteed
check(
    'Up 14, 5:00 left Q4, own 25',
    dict(score_differential=14, quarter=4, seconds_remaining=300,
         yardline_100=75, down=1, ydstogo=10,
         offense_timeouts=2, defense_timeouts=2,
         is_overtime=0, overtime_possession_number=0),
    expected_lo=0.90, expected_hi=1.00,
    note='Two-score lead with a full drive needed — should be >90%'
)

=== Blowout situations ===

✓ PASS  100.0%  expected [97%–100%]    Up 28, 2:00 left Q4, own 20
        note: Should be >97% — three score lead, clock nearly dead
✓ PASS    0.0%  expected [0%–3%]       Down 28, 2:00 left Q4, own 20
        note: Should be <3% — need three scores in two minutes, essentially impossible
✓ PASS   99.5%  expected [90%–100%]    Up 14, 5:00 left Q4, own 25
        note: Two-score lead with a full drive needed — should be >90%


0.9945945739746094

---
## 2. Coin-flip situations
These should be near 50% — the model should show no strong lean.

In [6]:
print('=== Near-50% situations ===\n')

# Tied at the start of Q3, own 25
check(
    'Tied, start of Q3, own 25, 1st & 10',
    dict(score_differential=0, quarter=3, seconds_remaining=1800,
         yardline_100=75, down=1, ydstogo=10,
         offense_timeouts=3, defense_timeouts=3,
         is_overtime=0, overtime_possession_number=0),
    expected_lo=0.45, expected_hi=0.55,
    note='Perfectly symmetric state — should be ~50%'
)

# Tied at halftime, receiving team
check(
    'Tied at halftime, own 25',
    dict(score_differential=0, quarter=2, seconds_remaining=1,
         yardline_100=75, down=1, ydstogo=10,
         offense_timeouts=3, defense_timeouts=3,
         is_overtime=0, overtime_possession_number=0),
    expected_lo=0.44, expected_hi=0.56,
    note='End of first half, tied — model does not know who gets second-half kick, so ~50%'
)

# Down 3, opponent 40, Q3, 10 min left — close but slightly behind
check(
    'Down 3, opp 40, Q3 10:00',
    dict(score_differential=-3, quarter=3, seconds_remaining=1500,
         yardline_100=40, down=1, ydstogo=10,
         offense_timeouts=3, defense_timeouts=3,
         is_overtime=0, overtime_possession_number=0),
    expected_lo=0.38, expected_hi=0.52,
    note='Down a field goal with plenty of time — slight deficit but very much in it'
)

=== Near-50% situations ===

✓ PASS   53.9%  expected [45%–55%]     Tied, start of Q3, own 25, 1st & 10
        note: Perfectly symmetric state — should be ~50%
✗ FAIL   56.3%  expected [44%–56%]     Tied at halftime, own 25
        note: End of first half, tied — model does not know who gets second-half kick, so ~50%
✓ PASS   43.3%  expected [38%–52%]     Down 3, opp 40, Q3 10:00
        note: Down a field goal with plenty of time — slight deficit but very much in it


0.43272119760513306

---
## 3. Late-game pressure situations
Testing urgency — the model should ramp WP down as time dwindles with a deficit.

In [7]:
print('=== Late-game pressure ===\n')

# Down 7, two-minute drill, own 20, 2 timeouts
check(
    'Down 7, 2:00 left Q4, own 20, 2 TOs',
    dict(score_differential=-7, quarter=4, seconds_remaining=120,
         yardline_100=80, down=1, ydstogo=10,
         offense_timeouts=2, defense_timeouts=0,
         is_overtime=0, overtime_possession_number=0),
    expected_lo=0.12, expected_hi=0.28,
    note='Two-minute drill: need a TD + stop or FG + stop + OT. Legitimate but hard — expect 15–25%'
)

# Down 3, FG range, 0:45 left, no TOs
check(
    'Down 3, opp 28 (FG range), 0:45 Q4, no TOs',
    dict(score_differential=-3, quarter=4, seconds_remaining=45,
         yardline_100=28, down=1, ydstogo=10,
         offense_timeouts=0, defense_timeouts=3,
         is_overtime=0, overtime_possession_number=0),
    expected_lo=0.28, expected_hi=0.48,
    note='In FG range to tie with 45s left — real shot but need to get closer or spike ball to kick'
)

# Up 3, opponent has ball at own 20, 1:30 left, they have 3 TOs
check(
    'Up 3, DEF has ball at own 20, 1:30 left, opponent 3 TOs — from DEFENDER view (we have ball at opp 20)',
    dict(score_differential=3, quarter=4, seconds_remaining=90,
         yardline_100=75, down=1, ydstogo=10,
         offense_timeouts=3, defense_timeouts=3,
         is_overtime=0, overtime_possession_number=0),
    expected_lo=0.55, expected_hi=0.72,
    note='Slight lead, clock ticking — leading team favoured but not comfortable'
)

# Down 14, Q4 10 min left — comeback window
check(
    'Down 14, Q4 10:00 left, own 25',
    dict(score_differential=-14, quarter=4, seconds_remaining=600,
         yardline_100=75, down=1, ydstogo=10,
         offense_timeouts=3, defense_timeouts=3,
         is_overtime=0, overtime_possession_number=0),
    expected_lo=0.07, expected_hi=0.20,
    note='Down two scores with 10 min: possible but unlikely — expect 10–18%'
)

=== Late-game pressure ===

✓ PASS   12.8%  expected [12%–28%]     Down 7, 2:00 left Q4, own 20, 2 TOs
        note: Two-minute drill: need a TD + stop or FG + stop + OT. Legitimate but hard — expect 15–25%
✓ PASS   37.2%  expected [28%–48%]     Down 3, opp 28 (FG range), 0:45 Q4, no TOs
        note: In FG range to tie with 45s left — real shot but need to get closer or spike ball to kick
✗ FAIL   86.4%  expected [55%–72%]     Up 3, DEF has ball at own 20, 1:30 left, opponent 3 TOs — from DEFENDER view (we have ball at opp 20)
        note: Slight lead, clock ticking — leading team favoured but not comfortable
✗ FAIL    5.4%  expected [7%–20%]      Down 14, Q4 10:00 left, own 25
        note: Down two scores with 10 min: possible but unlikely — expect 10–18%


0.053921569138765335

---
## 4. Field position extremes
Goal-to-go vs backed up to own end zone — does field position matter independently of score?

In [8]:
print('=== Field position extremes (same score/time) ===\n')

base = dict(score_differential=0, quarter=2, seconds_remaining=1800,
            down=1, ydstogo=10,
            offense_timeouts=3, defense_timeouts=3,
            is_overtime=0, overtime_possession_number=0)

check(
    'Tied Q3, 1st & goal at opp 5 (yardline=5)',
    {**base, 'yardline_100': 5},
    expected_lo=0.58, expected_hi=0.72,
    note='Red zone, likely to score — should be meaningfully above 50%'
)

check(
    'Tied Q3, 1st & 10 at own 25 (yardline=75) — neutral field position',
    {**base, 'yardline_100': 75},
    expected_lo=0.47, expected_hi=0.53,
    note='Neutral position — should be ~50%'
)

check(
    'Tied Q3, 1st & 10 backed up at own 1 (yardline=99)',
    {**base, 'yardline_100': 99},
    expected_lo=0.38, expected_hi=0.52,
    note='Pinned deep — safety risk, limited options. Modest disadvantage vs neutral'
)

print('\n  → WP should increase as yardline_100 decreases (closer to scoring)')

=== Field position extremes (same score/time) ===

✓ PASS   64.1%  expected [58%–72%]     Tied Q3, 1st & goal at opp 5 (yardline=5)
        note: Red zone, likely to score — should be meaningfully above 50%
✗ FAIL   53.9%  expected [47%–53%]     Tied Q3, 1st & 10 at own 25 (yardline=75) — neutral field position
        note: Neutral position — should be ~50%
✗ FAIL   53.9%  expected [38%–52%]     Tied Q3, 1st & 10 backed up at own 1 (yardline=99)
        note: Pinned deep — safety risk, limited options. Modest disadvantage vs neutral

  → WP should increase as yardline_100 decreases (closer to scoring)


---
## 5. Overtime scenarios
2023+ playoffs / 2025+ regular season rules: both teams are guaranteed a possession.
Set `guaranteed_possession=1` for these games.

In [9]:
print('=== Overtime scenarios ===\n')

# First OT possession, tied, own 25 — slight advantage since you have the ball
check(
    'OT 1st possession, tied, own 25 (old rules, no guaranteed poss)',
    dict(score_differential=0, quarter=5, seconds_remaining=600,
         yardline_100=75, down=1, ydstogo=10,
         offense_timeouts=2, defense_timeouts=2,
         is_overtime=1, overtime_possession_number=0,
         guaranteed_possession=0),
    expected_lo=0.50, expected_hi=0.65,
    note='Old rules (sudden death): scoring wins; possession advantage is meaningful'
)

check(
    'OT 1st possession, tied, own 25 (2023+ playoff / 2025+ reg, guaranteed poss)',
    dict(score_differential=0, quarter=5, seconds_remaining=600,
         yardline_100=75, down=1, ydstogo=10,
         offense_timeouts=2, defense_timeouts=2,
         is_overtime=1, overtime_possession_number=0,
         guaranteed_possession=1),
    expected_lo=0.50, expected_hi=0.60,
    note='New rules: opponent gets a possession too, so first-poss advantage is smaller'
)

# OT 2nd possession, down 3 (opponent kicked FG) — must score or game over
check(
    'OT 2nd possession, down 3, own 25 (guaranteed poss, must answer)',
    dict(score_differential=-3, quarter=5, seconds_remaining=400,
         yardline_100=75, down=1, ydstogo=10,
         offense_timeouts=2, defense_timeouts=2,
         is_overtime=1, overtime_possession_number=1,
         guaranteed_possession=1),
    expected_lo=0.22, expected_hi=0.42,
    note='Need at least a FG to stay alive — roughly 25-38% after accounting for drive success rate'
)

# OT 1st possession, up 3 in old-rules sudden death (kicked a FG)
# This shouldn't exist in old rules (game ends on FG) but tests the model handles it
check(
    'OT, up 3, opp ball at own 25 — from OUR perspective after trading FGs',
    dict(score_differential=3, quarter=5, seconds_remaining=200,
         yardline_100=75, down=1, ydstogo=10,
         offense_timeouts=2, defense_timeouts=2,
         is_overtime=1, overtime_possession_number=2,
         guaranteed_possession=1),
    expected_lo=0.58, expected_hi=0.80,
    note='Sudden death phase, up 3 — significant advantage'
)

=== Overtime scenarios ===

✓ PASS   56.3%  expected [50%–65%]     OT 1st possession, tied, own 25 (old rules, no guaranteed poss)
        note: Old rules (sudden death): scoring wins; possession advantage is meaningful
✓ PASS   56.3%  expected [50%–60%]     OT 1st possession, tied, own 25 (2023+ playoff / 2025+ reg, guaranteed poss)
        note: New rules: opponent gets a possession too, so first-poss advantage is smaller
✓ PASS   29.7%  expected [22%–42%]     OT 2nd possession, down 3, own 25 (guaranteed poss, must answer)
        note: Need at least a FG to stay alive — roughly 25-38% after accounting for drive success rate
✗ FAIL   86.4%  expected [58%–80%]     OT, up 3, opp ball at own 25 — from OUR perspective after trading FGs
        note: Sudden death phase, up 3 — significant advantage


0.8637738227844238

---
## 6. Time-of-game progression (same score, watching urgency change)
Up 7 from Q1 through Q4 — WP should increase as clock runs out.

In [10]:
print('=== Up 7 at different points in the game ===\n')
print('WP should rise as time remaining decreases (lead becomes safer)\n')

base = dict(score_differential=7, yardline_100=75, down=1, ydstogo=10,
            offense_timeouts=3, defense_timeouts=3,
            is_overtime=0, overtime_possession_number=0)

scenarios = [
    ('Start of Q1',  1, 3600),
    ('Start of Q2',  2, 2700),
    ('Start of Q3',  3, 1800),
    ('Start of Q4',  4,  900),
    ('Q4 10:00',     4,  600),
    ('Q4 5:00',      4,  300),
    ('Q4 2:00',      4,  120),
    ('Q4 0:30',      4,   30),
]

wps = []
for label, qtr, secs in scenarios:
    wp = model.predict_proba({**base, 'quarter': qtr, 'seconds_remaining': secs})
    wps.append(wp)
    bar = '█' * int(wp * 40)
    print(f'  {label:<18s}  {wp*100:5.1f}%  {bar}')

monotone = all(wps[i] <= wps[i+1] for i in range(len(wps)-1))
print(f'\n  Monotonically increasing: {"✓ YES" if monotone else "✗ NO — model not fully monotone"}')

=== Up 7 at different points in the game ===

WP should rise as time remaining decreases (lead becomes safer)

  Start of Q1          72.2%  ████████████████████████████
  Start of Q2          72.9%  █████████████████████████████
  Start of Q3          80.2%  ████████████████████████████████
  Start of Q4          83.5%  █████████████████████████████████
  Q4 10:00             86.4%  ██████████████████████████████████
  Q4 5:00              91.2%  ████████████████████████████████████
  Q4 2:00              93.1%  █████████████████████████████████████
  Q4 0:30              96.1%  ██████████████████████████████████████

  Monotonically increasing: ✓ YES


---
## 7. Full summary table

In [14]:
pip install jinja2

  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached markupsafe-3.0.3-cp313-cp313-win_amd64.whl.metadata (2.8 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
Using cached markupsafe-3.0.3-cp313-cp313-win_amd64.whl (15 kB)

   -------------------- ------------------- 1/2 [jinja2]
   -------------------- ------------------- 1/2 [jinja2]
   -------------------- ------------------- 1/2 [jinja2]
   ---------------------------------------- 2/2 [jinja2]

Note: you may need to restart the kernel to use updated packages.


In [15]:
import pandas as pd
import jinja2

scenarios = [
    # (label,                         score_diff, qtr, secs, yl,  is_ot)
    ('Blowout win (up 28, 2 min)',         28,  4,  120,  80, 0),
    ('Comfortable win (up 14, 5 min)',     14,  4,  300,  75, 0),
    ('Up 7, Q4 start',                     7,  4,  900,  75, 0),
    ('Up 3, Q4 5:00',                      3,  4,  300,  75, 0),
    ('Tied, Q3 start',                     0,  3, 1800,  75, 0),
    ('Down 3, Q4 5:00',                   -3,  4,  300,  75, 0),
    ('Down 7, 2-min drill',               -7,  4,  120,  80, 0),
    ('Down 14, Q4 10:00',                -14,  4,  600,  75, 0),
    ('Blowout loss (down 28, 2 min)',     -28,  4,  120,  80, 0),
    ('OT tied, 1st possession',            0,  5,  600,  75, 1),
    ('OT down 3, 2nd possession',         -3,  5,  400,  75, 1),
]

rows = []
for label, sd, qtr, secs, yl, is_ot in scenarios:
    state = dict(
        score_differential=sd, quarter=qtr, seconds_remaining=secs,
        yardline_100=yl, down=1, ydstogo=10,
        offense_timeouts=2, defense_timeouts=2,
        is_overtime=is_ot, overtime_possession_number=(1 if (is_ot and sd != 0) else 0),
        guaranteed_possession=1 if is_ot else 0,
    )
    wp = model.predict_proba(state)
    rows.append({'Scenario': label, 'Score diff': sd, 'Quarter': qtr,
                 'Secs left': secs, 'WP': f'{wp*100:.1f}%'})

df = pd.DataFrame(rows)
display(df.style.set_properties(**{'text-align': 'left'}))

,Scenario,Score diff,Quarter,Secs left,WP
0,"Blowout win (up 28, 2 min)",28,4,120,100.0%
1,"Comfortable win (up 14, 5 min)",14,4,300,99.5%
2,"Up 7, Q4 start",7,4,900,86.4%
3,"Up 3, Q4 5:00",3,4,300,72.2%
4,"Tied, Q3 start",0,3,1800,55.5%
5,"Down 3, Q4 5:00",-3,4,300,29.7%
6,"Down 7, 2-min drill",-7,4,120,12.8%
7,"Down 14, Q4 10:00",-14,4,600,5.4%
8,"Blowout loss (down 28, 2 min)",-28,4,120,0.0%
9,"OT tied, 1st possession",0,5,600,56.3%
